# NBA Live Win Probability — Feature Engineering

This notebook builds the final feature set for the live NBA win probability model. The features are intentionally limited to signals that improved the logistic benchmark during temporal validation:

1. game-state baseline features: score, time, possession, overtime, period;
2. estimated possessions remaining;
3. fourth-quarter bonus indicators;
4. late-game score-state buckets;
5. a blended pregame team-strength prior.

Features that were tested but did not improve the benchmark, including raw rest, back-to-back indicators, raw momentum, and shot-quality/luck variants, are excluded from the final pipeline. 


In [1]:
# ============================================================
# Imports and configuration
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# Update this path if your data file lives somewhere else in the repo.
DATA_PATH = Path("../data/nba_win_prob_slim.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("nba_win_prob_slim.csv")

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Data path:", DATA_PATH)


Data path: nba_win_prob_slim.csv


## 1. Load and validate modeling data

The notebook expects the slim play-by-play modeling file produced by the data-building notebook. The split is temporal by game date, so later games are held out as the test set.


In [2]:
# ============================================================
# Load data
# ============================================================

df = pd.read_csv(DATA_PATH, low_memory=False)

df["game_id"] = df["game_id"].astype(str).str.zfill(10)
df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")

required_columns = [
    "game_id",
    "game_date",
    "home_team_won",
    "period",
    "is_overtime",
    "seconds_remaining",
    "score_diff",
    "home_possession",
    "home_fouls",
    "away_fouls",
    "diff_net_rtg_lastseason",
    "diff_net_rtg_last20",
    "diff_net_rtg_last10",
]

missing = sorted(set(required_columns) - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Games: {df['game_id'].nunique():,}")
print(f"Date range: {df['game_date'].min().date()} to {df['game_date'].max().date()}")


Rows: 1,661,962
Columns: 66
Games: 3,855
Date range: 2023-10-24 to 2026-04-12


## 2. Feature engineering functions

The functions below are small and composable. Each group corresponds to a feature family that survived ablation testing.


In [3]:
# ============================================================
# Feature engineering helpers
# ============================================================

def add_baseline_game_state_features(data: pd.DataFrame) -> pd.DataFrame:
    """Add simple score/time transformations used by the baseline logit model."""
    out = data.copy()

    out["abs_score_diff"] = out["score_diff"].abs()
    out["minutes_remaining"] = out["seconds_remaining"] / 60.0

    out["score_x_time"] = out["score_diff"] * out["minutes_remaining"]
    out["abs_score_x_time"] = out["abs_score_diff"] * out["minutes_remaining"]

    # The +0.5 avoids extreme ratios at the very end of games.
    out["score_per_minute_remaining"] = out["score_diff"] / (out["minutes_remaining"] + 0.5)

    return out


def add_possession_remaining_features(data: pd.DataFrame, avg_seconds_per_possession: float = 28.8) -> pd.DataFrame:
    """Estimate remaining possessions and convert score margin into possession-scale features."""
    out = data.copy()

    out["possessions_remaining_est"] = (out["seconds_remaining"] / avg_seconds_per_possession).clip(lower=0)

    out["score_per_possession_remaining"] = (
        out["score_diff"] / (out["possessions_remaining_est"] + 1.0)
    )
    out["abs_score_per_possession_remaining"] = (
        out["score_diff"].abs() / (out["possessions_remaining_est"] + 1.0)
    )

    out["score_x_possessions_remaining"] = out["score_diff"] * out["possessions_remaining_est"]
    out["abs_score_x_possessions_remaining"] = out["score_diff"].abs() * out["possessions_remaining_est"]

    return out


def add_q4_bonus_features(data: pd.DataFrame) -> pd.DataFrame:
    """Estimate fourth-quarter team-foul bonus status from cumulative team foul counts."""
    out = data.copy()

    out = out.sort_values(
        ["game_id", "period", "seconds_remaining"],
        ascending=[True, True, False],
    ).reset_index(drop=True)

    q4_start_fouls = (
        out[out["period"] == 4]
        .groupby("game_id")[["home_fouls", "away_fouls"]]
        .first()
        .rename(columns={
            "home_fouls": "home_fouls_start_q4",
            "away_fouls": "away_fouls_start_q4",
        })
    )

    out = out.drop(columns=["home_fouls_start_q4", "away_fouls_start_q4"], errors="ignore")
    out = out.merge(q4_start_fouls, on="game_id", how="left")

    out["home_fouls_q4"] = np.where(
        out["period"] == 4,
        out["home_fouls"] - out["home_fouls_start_q4"],
        0,
    )
    out["away_fouls_q4"] = np.where(
        out["period"] == 4,
        out["away_fouls"] - out["away_fouls_start_q4"],
        0,
    )

    out["home_fouls_q4"] = out["home_fouls_q4"].fillna(0).clip(lower=0)
    out["away_fouls_q4"] = out["away_fouls_q4"].fillna(0).clip(lower=0)

    # If away has 5+ Q4 team fouls, the home offense is in the bonus, and vice versa.
    out["home_offense_bonus_q4"] = ((out["period"] == 4) & (out["away_fouls_q4"] >= 5)).astype(int)
    out["away_offense_bonus_q4"] = ((out["period"] == 4) & (out["home_fouls_q4"] >= 5)).astype(int)

    out["current_offense_has_bonus_q4"] = np.where(
        out["home_possession"] == 1,
        out["home_offense_bonus_q4"],
        out["away_offense_bonus_q4"],
    )

    return out


def add_late_game_score_state_features(data: pd.DataFrame) -> pd.DataFrame:
    """Add interpretable late-Q4 score buckets for one-, two-, and three-possession states."""
    out = data.copy()

    out["late_q4"] = ((out["period"] == 4) & (out["seconds_remaining"] <= 300)).astype(int)

    out["home_up_1_to_3_late_q4"] = ((out["score_diff"].between(1, 3)) & (out["late_q4"] == 1)).astype(int)
    out["home_down_1_to_3_late_q4"] = ((out["score_diff"].between(-3, -1)) & (out["late_q4"] == 1)).astype(int)

    out["home_up_4_to_6_late_q4"] = ((out["score_diff"].between(4, 6)) & (out["late_q4"] == 1)).astype(int)
    out["home_down_4_to_6_late_q4"] = ((out["score_diff"].between(-6, -4)) & (out["late_q4"] == 1)).astype(int)

    out["home_up_7_to_9_late_q4"] = ((out["score_diff"].between(7, 9)) & (out["late_q4"] == 1)).astype(int)
    out["home_down_7_to_9_late_q4"] = ((out["score_diff"].between(-9, -7)) & (out["late_q4"] == 1)).astype(int)

    out["tied_late_q4"] = ((out["score_diff"] == 0) & (out["late_q4"] == 1)).astype(int)

    return out


def add_blended_team_strength(data: pd.DataFrame) -> pd.DataFrame:
    """Blend stable preseason/team-quality signal with recent-form signal."""
    out = data.copy()

    out["diff_net_rtg_blend_70_20_10"] = (
        0.70 * out["diff_net_rtg_lastseason"]
        + 0.20 * out["diff_net_rtg_last20"]
        + 0.10 * out["diff_net_rtg_last10"]
    )

    return out


def build_final_features(data: pd.DataFrame) -> pd.DataFrame:
    """Run the final feature engineering pipeline."""
    out = data.copy()
    out = add_baseline_game_state_features(out)
    out = add_possession_remaining_features(out)
    out = add_q4_bonus_features(out)
    out = add_late_game_score_state_features(out)
    out = add_blended_team_strength(out)
    return out


## 3. Build final feature matrix

The final feature list is deliberately compact. Each group either improved overall log loss or improved relevant late-game subsets without hurting the full test set.


In [4]:
# ============================================================
# Build final features
# ============================================================

df_features = build_final_features(df)

BASELINE_GAME_STATE_FEATURES = [
    "period",
    "is_overtime",
    "seconds_remaining",
    "minutes_remaining",
    "score_diff",
    "abs_score_diff",
    "home_possession",
    "score_x_time",
    "abs_score_x_time",
    "score_per_minute_remaining",
]

TEAM_STRENGTH_FEATURES = [
    "diff_net_rtg_blend_70_20_10",
]

Q4_BONUS_FEATURES = [
    "home_offense_bonus_q4",
    "away_offense_bonus_q4",
    "current_offense_has_bonus_q4",
]

POSSESSION_REMAINING_FEATURES = [
    "possessions_remaining_est",
    "score_per_possession_remaining",
    "abs_score_per_possession_remaining",
    "score_x_possessions_remaining",
    "abs_score_x_possessions_remaining",
]

LATE_SCORE_STATE_FEATURES = [
    "home_up_1_to_3_late_q4",
    "home_down_1_to_3_late_q4",
    "home_up_4_to_6_late_q4",
    "home_down_4_to_6_late_q4",
    "home_up_7_to_9_late_q4",
    "home_down_7_to_9_late_q4",
    "tied_late_q4",
]

FINAL_FEATURES = (
    BASELINE_GAME_STATE_FEATURES
    + TEAM_STRENGTH_FEATURES
    + Q4_BONUS_FEATURES
    + POSSESSION_REMAINING_FEATURES
    + LATE_SCORE_STATE_FEATURES
)

missing_final = sorted(set(FINAL_FEATURES) - set(df_features.columns))
if missing_final:
    raise ValueError(f"Final feature columns missing after engineering: {missing_final}")

print(f"Final feature count: {len(FINAL_FEATURES)}")
display(pd.DataFrame({"feature": FINAL_FEATURES}))


Final feature count: 26


,feature
0,period
1,is_overtime
2,seconds_remaining
3,minutes_remaining
4,score_diff
5,abs_score_diff
6,home_possession
7,score_x_time
8,abs_score_x_time
9,score_per_minute_remaining


## 4. Temporal validation utilities

The benchmark uses a temporal split by game date. This is more realistic than a random row split because the model is evaluated on future games.


In [5]:
# ============================================================
# Validation helpers
# ============================================================

def get_temporal_train_test_games(data: pd.DataFrame, test_size: float = 0.20):
    games = (
        data[["game_id", "game_date"]]
        .drop_duplicates("game_id")
        .dropna(subset=["game_date"])
        .sort_values("game_date")
        .reset_index(drop=True)
    )

    cutoff = int(len(games) * (1 - test_size))
    train_games = games.iloc[:cutoff]["game_id"]
    test_games = games.iloc[cutoff:]["game_id"]

    print(f"Train games: {train_games.nunique():,} | {games.iloc[:cutoff]['game_date'].min().date()} to {games.iloc[:cutoff]['game_date'].max().date()}")
    print(f"Test games:  {test_games.nunique():,} | {games.iloc[cutoff:]['game_date'].min().date()} to {games.iloc[cutoff:]['game_date'].max().date()}")

    return train_games, test_games


def train_logit_model(data: pd.DataFrame, features: list[str], train_games, test_games, C: float = 1.0):
    model_cols = [
        "game_id",
        "game_date",
        "home_team_won",
        "period",
        "seconds_remaining",
        "score_diff",
    ] + features

    model_df = data[list(dict.fromkeys(model_cols))].copy()
    model_df = model_df.dropna(subset=["home_team_won"])

    train_mask = model_df["game_id"].isin(train_games)
    test_mask = model_df["game_id"].isin(test_games)

    X_train = model_df.loc[train_mask, features].copy()
    y_train = model_df.loc[train_mask, "home_team_won"].astype(int)

    X_test = model_df.loc[test_mask, features].copy()
    y_test = model_df.loc[test_mask, "home_team_won"].astype(int)

    train_medians = X_train.median(numeric_only=True)
    X_train = X_train.fillna(train_medians)
    X_test = X_test.fillna(train_medians)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            C=C,
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
        )),
    ])

    model.fit(X_train, y_train)
    pred = model.predict_proba(X_test)[:, 1]

    results = model_df.loc[test_mask, ["game_id", "game_date", "period", "seconds_remaining", "score_diff"]].copy()
    results["actual"] = y_test.values
    results["pred_home_win_prob"] = pred

    return model, results


def score_predictions(results: pd.DataFrame, subset_name: str = "all") -> dict:
    y_true = results["actual"]
    y_pred = results["pred_home_win_prob"]

    return {
        "subset": subset_name,
        "rows": len(results),
        "games": results["game_id"].nunique(),
        "log_loss": log_loss(y_true, y_pred),
        "brier_score": brier_score_loss(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_pred) if y_true.nunique() > 1 else np.nan,
        "actual_home_win_rate": y_true.mean(),
        "avg_pred_home_win_prob": y_pred.mean(),
    }


## 5. Ablation check for retained feature groups

This section documents why the retained features are in the final pipeline. The final model uses the blended team-strength prior plus the game-context features that survived earlier ablations.


In [7]:
# ============================================================
# Ablation over retained feature groups
# ============================================================

feature_sets = {
    "baseline_lastseason": (
        BASELINE_GAME_STATE_FEATURES
        + ["diff_net_rtg_lastseason"]
    ),
    "baseline_blended_strength": (
        BASELINE_GAME_STATE_FEATURES
        + TEAM_STRENGTH_FEATURES
    ),
    "plus_q4_bonus": (
        BASELINE_GAME_STATE_FEATURES
        + TEAM_STRENGTH_FEATURES
        + Q4_BONUS_FEATURES
    ),
    "plus_possessions": (
        BASELINE_GAME_STATE_FEATURES
        + TEAM_STRENGTH_FEATURES
        + Q4_BONUS_FEATURES
        + POSSESSION_REMAINING_FEATURES
    ),
    "final_logit_features": FINAL_FEATURES,
}

train_games, test_games = get_temporal_train_test_games(df_features, test_size=TEST_SIZE)

models = {}
results_by_model = {}
summary_rows = []

for name, features in feature_sets.items():
    model, results = train_logit_model(
        data=df_features,
        features=features,
        train_games=train_games,
        test_games=test_games,
        C=1.0,
    )

    models[name] = model
    results_by_model[name] = results

    metrics = score_predictions(results, subset_name="all")
    metrics["model"] = name
    metrics["num_features"] = len(features)
    summary_rows.append(metrics)

ablation_summary = pd.DataFrame(summary_rows)

cols = [
    "model",
    "num_features",
    "rows",
    "games",
    "log_loss",
    "brier_score",
    "roc_auc",
    "actual_home_win_rate",
    "avg_pred_home_win_prob",
]

display(ablation_summary[cols].sort_values("log_loss"))


Train games: 3,084 | 2023-10-24 to 2025-12-27
Test games:  771 | 2025-12-27 to 2026-04-12


,model,num_features,rows,games,log_loss,brier_score,roc_auc,actual_home_win_rate,avg_pred_home_win_prob
4,final_logit_features,26,334468,771,0.421873,0.139723,0.881991,0.555949,0.548468
3,plus_possessions,19,334468,771,0.421960,0.139758,0.881902,0.555949,0.548500
2,plus_q4_bonus,14,334468,771,0.421968,0.139764,0.881881,0.555949,0.548501
1,baseline_blended_strength,11,334468,771,0.422007,0.139780,0.881846,0.555949,0.548557
0,baseline_lastseason,11,334468,771,0.423562,0.140353,0.880853,0.555949,0.548205


## 6. Final model subset evaluation

The final features are especially designed to improve late-game and close-game states, so the benchmark is also checked on those subsets.


In [8]:
# ============================================================
# Final model subset evaluation
# ============================================================

final_results = results_by_model["final_logit_features"].copy()

subsets = {
    "all": final_results,
    "first_half": final_results[final_results["period"] <= 2],
    "q4_only": final_results[final_results["period"] == 4],
    "late_q4": final_results[
        (final_results["period"] == 4)
        & (final_results["seconds_remaining"] <= 300)
    ],
    "close_q4": final_results[
        (final_results["period"] == 4)
        & (final_results["score_diff"].abs() <= 10)
    ],
    "clutch_q4": final_results[
        (final_results["period"] == 4)
        & (final_results["seconds_remaining"] <= 300)
        & (final_results["score_diff"].abs() <= 5)
    ],
}

subset_summary = pd.DataFrame([
    score_predictions(subset, subset_name=name)
    for name, subset in subsets.items()
    if len(subset) >= 100
])

display(subset_summary[
    [
        "subset",
        "rows",
        "games",
        "log_loss",
        "brier_score",
        "roc_auc",
        "actual_home_win_rate",
        "avg_pred_home_win_prob",
    ]
].sort_values("subset"))


,subset,rows,games,log_loss,brier_score,roc_auc,actual_home_win_rate,avg_pred_home_win_prob
0,all,334468,771,0.421873,0.139723,0.881991,0.555949,0.548468
4,close_q4,39528,512,0.422774,0.138269,0.884889,0.540427,0.519016
5,clutch_q4,10515,321,0.450145,0.147616,0.867044,0.554351,0.528754
1,first_half,166157,771,0.538464,0.181462,0.801870,0.556064,0.550890
3,late_q4,37039,771,0.154328,0.048531,0.985294,0.556899,0.549665
2,q4_only,83640,771,0.221594,0.070237,0.969610,0.556074,0.546396


## 7. Save engineered modeling data

The saved file contains the target, game identifiers, final model features, and a small set of metadata columns useful for downstream model evaluation. This becomes the input to the XGBoost / boosted-tree notebook.


In [9]:
# ============================================================
# Save final feature matrix and feature list
# ============================================================

OUTPUT_DIR = Path("../data/processed")
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path("processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata_columns = [
    "game_id",
    "game_date",
    "home_team_won",
    "period",
    "seconds_remaining",
    "score_diff",
]

save_columns = metadata_columns + FINAL_FEATURES
save_columns = list(dict.fromkeys(save_columns))

feature_matrix_path = OUTPUT_DIR / "nba_live_wp_final_features.csv"
feature_list_path = OUTPUT_DIR / "final_logit_features.json"

final_feature_matrix = df_features[save_columns].copy()
final_feature_matrix.to_csv(feature_matrix_path, index=False)

with open(feature_list_path, "w", encoding="utf-8") as f:
    json.dump(FINAL_FEATURES, f, indent=2)

print(f"Saved feature matrix: {feature_matrix_path}")
print(f"Saved feature list:   {feature_list_path}")
print(f"Shape: {final_feature_matrix.shape}")


Saved feature matrix: processed\nba_live_wp_final_features.csv
Saved feature list:   processed\final_logit_features.json
Shape: (1661962, 29)


## Summary

The final logistic feature set is intentionally compact and interpretable. The largest improvement came from replacing the simple previous-season team-strength prior with a blended prior:

`70% previous-season net rating + 20% last-20 net rating + 10% last-10 net rating`.

Small but consistent improvements also came from possessions remaining, Q4 bonus context, and late-game score-state buckets. Rejected features are not included in this notebook, but previous testing revealed that rest/B2B, momentum, and shot-quality variants did not improve the final temporal-test benchmark.
